# Homework: Monitoring a Trivia Agent with Logfire

This notebook follows the step-by-step style of `logfire_data_export.ipynb`. It collects the preparation code and verified outputs for the trivia monitoring homework. Agent implementation and trace-analysis sections will be added when the questions are provided.

## Preparation: install dependencies

For a **new project**, run in a terminal:

```bash
uv init
uv add requests pydantic-ai logfire
```

This repository is already initialized, so `uv init` is unnecessary. The dependency command has been run here. Select this project's `.venv` Python interpreter when opening the notebook.

In [1]:
from pathlib import Path
import sys

# Support launching this notebook from the repository root or notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "trivia_tools.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "trivia_tools.py").exists():
    raise FileNotFoundError("Open the notebook from the project root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import dotenv
dotenv.load_dotenv(PROJECT_ROOT / ".env")
print("Project imports and environment loaded")

Project imports and environment loaded


## Provided tools

`trivia_tools.py` contains the supplied `TriviaTools` class:

- `get_categories()` requests the available category IDs and names.
- `get_questions(amount, category, difficulty)` requests multiple-choice questions and decodes HTML entities.

The question tool returns correct and incorrect answers to the agent. The future quiz agent should present shuffled choices without revealing the answer before the player responds. Calling the categories tool directly does not require an LLM, an OpenAI key, or Logfire authentication.

In [2]:
from trivia_tools import TriviaTools

tools = TriviaTools()
categories_text = tools.get_categories()
print(categories_text)

9: General Knowledge
10: Entertainment: Books
11: Entertainment: Film
12: Entertainment: Music
13: Entertainment: Musicals & Theatres
14: Entertainment: Television
15: Entertainment: Video Games
16: Entertainment: Board Games
17: Science & Nature
18: Science: Computers
19: Science: Mathematics
20: Mythology
21: Sports
22: Geography
23: History
24: Politics
25: Art
26: Celebrities
27: Animals
28: Vehicles
29: Entertainment: Comics
30: Science: Gadgets
31: Entertainment: Japanese Anime & Manga
32: Entertainment: Cartoon & Animations


## Verify the categories

The preparation instructions expect 24 categories. Count the actual lines returned by the live API and verify that expectation.

In [3]:
categories = [line for line in categories_text.splitlines() if line.strip()]
category_count = len(categories)
print(f"Returned categories: {category_count}")
assert category_count == 24, f"Expected 24 categories, received {category_count}"
print("Preparation check passed")

Returned categories: 24
Preparation check passed


## Monitoring questions

The preparation is complete when the cells above execute successfully. Subsequent homework questions will add the trivia agent, monitored quiz sessions, Logfire queries, and analysis here. No monitoring results or trace IDs have been assumed.

## Question 1: Create and run the agent

Create `trivia_agent.py` with the quizmaster instructions and both provided tools. The terminal runner preserves message history so the agent can continue the quiz across player answers. It stops when the player enters `quit`, `exit`, or `stop`.

```bash
uv run python trivia_agent.py
```

For this notebook run, start a session with **Let's play 3 medium questions from History**. Inspect the actual messages to determine the first tool called, rather than assuming that the order of tools in the agent's list determines their call order.

### Agent source

```python
instructions = """You are a trivia quizmaster. When asked to play trivia:
1. Use the available tools to fetch trivia questions
2. Ask the player one question at a time with multiple choice options
3. Wait for their answer before moving to the next question
4. When the player answers, explain why the correct answer is correct - add interesting context and facts
5. After all questions, give the final score
"""

trivia_tools = TriviaTools()

agent = Agent(
    'openai:gpt-4o-mini',
    name='agent',
    tools=[trivia_tools.get_categories, trivia_tools.get_questions],
    instructions=instructions,
)
import sys
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent

from trivia_tools import TriviaTools

load_dotenv(Path(__file__).with_name(".env"))

instructions = """You are a trivia quizmaster. When asked to play trivia:
1. Use the available tools to fetch trivia questions
2. Ask the player one question at a time with multiple choice options
3. Wait for their answer before moving to the next question
4. When the player answers, explain why the correct answer is correct - add interesting context and facts
5. After all questions, give the final score
Do not reveal correct answers before the player responds.
"""

trivia_tools = TriviaTools()
agent = Agent(
    "openai:gpt-4o-mini",
    name="agent",
    tools=[trivia_tools.get_categories, trivia_tools.get_questions],
    instructions=instructions,
)


if __name__ == "__main__":
    history = []
    prompt = " ".join(sys.argv[1:]) or input("You (or 'quit'): ")
    while prompt.strip().lower() not in {"quit", "exit", "stop"}:
        result = agent.run_sync(prompt, message_history=history)
        history.extend(result.new_messages())
        print(f"Quizmaster: {result.output}")
        prompt = input("You (or 'quit'): ")

```

In [4]:
from trivia_agent import agent
from utils import collect_tools

quiz_prompt = "Let's play 3 medium questions from History"
quiz_result = await agent.run(quiz_prompt)
quiz_history = quiz_result.new_messages()
quiz_tool_calls = collect_tools(quiz_history)

print("Prompt:", quiz_prompt)
print("Tool calls in order:", [call.name for call in quiz_tool_calls])
assert quiz_tool_calls, "Expected the agent to fetch trivia questions"
print("First tool called:", quiz_tool_calls[0].name)
print("\nQuizmaster:\n" + quiz_result.output)

17:57:24.530 agent run
17:57:24.532   chat gpt-4o-mini


Logfire project URL: ]8;id=304113;https://logfire-us.pydantic.dev/priyagarnepudi/priya-logfire-test\https://logfire-us.pydantic.dev/priyagarnepudi/priya-logfire-test]8;;\

17:57:25.654   running 1 tool
17:57:25.655     running tool: get_categories


17:57:25.952   chat gpt-4o-mini


17:57:27.042   running 1 tool
17:57:27.042     running tool: get_questions


17:57:27.373   chat gpt-4o-mini


Prompt: Let's play 3 medium questions from History
Tool calls in order: ['get_categories', 'get_questions']
First tool called: get_categories

Quizmaster:
Great! Let's get started with your first question.

**Question 1:** Which U.S. president took part in the Potsdam Conference, where the Allies reached a peace settlement with Germany?
  
A) Dwight D. Eisenhower  
B) Franklin D. Roosevelt  
C) Harry S. Truman  
D) Herbert Hoover  

Please respond with the letter of your answer.


The output above is the observed answer for this run. The tool list registers available tools; it does not force a particular execution order. The agent's first response should present only the first question and wait for the player's answer. `quiz_history` retains the tool responses and conversation for subsequent turns.

## Question 2: Set up monitoring

`trivia_agent.py` now loads `.env`, configures Logfire, and instruments Pydantic AI before creating the agent:

```python
import os
import logfire

logfire.configure(
    service_name="trivia-agent",
    token=os.getenv("LOGFIRE_TOKEN") or os.getenv("LOGFIRE_WRITE_TOKEN"),
    send_to_logfire="if-token-present",
)
logfire.instrument_pydantic_ai()
```

This follows the [Pydantic AI Logfire integration](https://pydantic.dev/docs/ai/integrations/logfire/). `if-token-present` allows local use without authentication; cloud export requires a write token or existing project credentials.

### Connect your Logfire project

1. Sign in to Logfire and create or select a project for this homework.
2. Create a write token and save it locally in `.env` as `LOGFIRE_TOKEN=your_write_token` or `LOGFIRE_WRITE_TOKEN=your_write_token`. Do not put the real token into this notebook or commit it.
3. If querying from this notebook, use a read token from the same project as `LOGFIRE_READ_TOKEN`.
4. Restart the kernel or terminal process, then run the agent once. Open the project's Live view and inspect the parent agent span.

A write token was subsequently supplied in `.env` as `LOGFIRE_WRITE_TOKEN`. The notebook was executed successfully with that token configured for cloud export. The following cell also captures actual OpenTelemetry spans locally, independently verifying the span name. A follow-up cloud query using the existing `LOGFIRE_READ_TOKEN` returned **Invalid token**; cloud records have not been independently verified. Use a valid read token from the same project or inspect the signed-in dashboard.

In [5]:
# Capture an actual instrumented run locally without requiring cloud credentials.
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

span_exporter = InMemorySpanExporter()
span_processor = SimpleSpanProcessor(span_exporter)
otel_trace.get_tracer_provider().add_span_processor(span_processor)

monitored_result = await agent.run(
    "Please repeat the current question and its choices without revealing the answer.",
    message_history=quiz_history,
)
import logfire
logfire.force_flush()

finished_spans = span_exporter.get_finished_spans()
root_spans = [span for span in finished_spans if span.parent is None]
assert root_spans, "Expected a top-level instrumented agent span"
print("Top-level span_name:", root_spans[0].name)
print("Local span names:", [span.name for span in finished_spans])
print("\nQuizmaster:\n" + monitored_result.output)
# Stop collecting after this run; no global tracing configuration is replaced.
span_exporter.clear()

17:57:29.608 agent run
17:57:29.609   chat gpt-4o-mini


Top-level span_name: agent run
Local span names: ['chat gpt-4o-mini', 'agent run']

Quizmaster:
Sure! Here’s the current question along with the choices:

**Question 1:** Which U.S. president took part in the Potsdam Conference, where the Allies reached a peace settlement with Germany?

A) Dwight D. Eisenhower  
B) Franklin D. Roosevelt  
C) Harry S. Truman  
D) Herbert Hoover  

Please respond with the letter of your answer.


### Answer and dashboard query

The installed Pydantic AI version emits **`agent run`** as the top-level agent span name. The executed cell above verifies the name from an actual run's exported spans. The agent's `name="agent"` and the service name `trivia-agent` are different fields, not the `span_name`.

After configuring cloud export and running the agent, verify in the Logfire dashboard with:

```sql
SELECT trace_id, span_name, start_timestamp
FROM records
WHERE service_name = 'trivia-agent'
  AND span_name = 'agent run'
ORDER BY start_timestamp DESC
LIMIT 10;
```

The instrumented run completed with the write token configured. Cloud verification remains pending because the existing read token was rejected and the browser requires sign-in. Span names may differ with other framework versions.

## Question 3: Play a full session

Start a five-question quiz and answer each question before entering `stop`.

The expected answer is **6 separate agent-run traces**:

1. The initial prompt fetches questions and presents question 1.
2. Answer 1 is graded and question 2 is presented.
3. Answer 2 is graded and question 3 is presented.
4. Answer 3 is graded and question 4 is presented.
5. Answer 4 is graded and question 5 is presented.
6. Answer 5 is graded and the final score is returned.

`stop` is checked before another agent call, so it creates no additional agent-run trace. Each `agent.run_sync` invocation starts a new trace in this setup; carrying message history preserves the conversation without making these runs children of a shared session span.

The homework calls these spans `invoke_agent agent`. The currently installed version emitted `agent run` in Question 2; use the name visible in your dashboard. Six is the expected count for this interaction pattern, not a claimed dashboard observation. Extra conversational turns would increase it.

In [ ]:
def run(prompt):
    message_history = []

    while True:
        result = agent.run_sync(prompt, message_history=message_history)
        print(result.output)
        message_history = result.all_messages()

        prompt = input("You (write 'stop' to stop): ")
        if not prompt or prompt.lower().strip() == 'stop':
            break

The supplied runner is synchronous. Run it in a terminal Python session, or use the asynchronous notebook version below to avoid conflicts with Jupyter's active event loop. The interactive call is commented out so executing the entire notebook does not pause for player input.

In [ ]:
async def run_notebook_session(prompt):
    message_history = []
    run_count = 0
    while True:
        result = await agent.run(prompt, message_history=message_history)
        run_count += 1
        print(result.output)
        message_history = result.all_messages()
        prompt = input("You (write 'stop' to stop): ")
        if not prompt or prompt.lower().strip() == "stop":
            break
    print(f"Agent runs in this session: {run_count}")
    return message_history

# Uncomment to play interactively:
# session_messages = await run_notebook_session(
#     "Let's play 5 easy questions from Science & Nature"
# )

## Question 4: Group the session under one parent span

Wrap all turns in `logfire.span("trivia_session")`. This makes the separate agent runs children of one session span. In a notebook, use `await agent.run` because Jupyter already has an event loop.

The following reproducible session selects **A** for each of the five questions. It sums the input-token usage from each run (including all model requests within the run), and independently checks exported agent spans. No fixed token count is assumed: fetched questions, explanations, and accumulated message history change the total.

In [1]:
import sys
from pathlib import Path
import logfire
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

project_root = Path.cwd().resolve()
if not (project_root / "trivia_agent.py").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from trivia_agent import agent

session_exporter = InMemorySpanExporter()
otel_trace.get_tracer_provider().add_span_processor(SimpleSpanProcessor(session_exporter))

session_history = []
session_input_tokens = 0
session_output_tokens = 0
session_run_count = 0
prompt = "Let's play 5 easy questions from Science & Nature"

with logfire.span("trivia_session"):
    for turn in range(6):
        if turn:
            prompt = "A"
            print("You:", prompt)
        result = await agent.run(prompt, message_history=session_history)
        session_history = result.all_messages()
        session_run_count += 1
        session_input_tokens += result.usage().input_tokens or 0
        session_output_tokens += result.usage().output_tokens or 0
        print(f"Quizmaster (run {session_run_count}):\n{result.output}\n")

logfire.force_flush()
exported_spans = session_exporter.get_finished_spans()
parent_spans = [s for s in exported_spans if s.name == "trivia_session"]
assert len(parent_spans) == 1
parent_span = parent_spans[0]
agent_spans = [s for s in exported_spans if s.name in {"agent run", "invoke_agent agent"}]
assert len(agent_spans) == 6
assert all(s.parent and s.parent.span_id == parent_span.context.span_id for s in agent_spans)
span_input_tokens = sum(s.attributes.get("gen_ai.usage.input_tokens", 0) for s in agent_spans)
assert span_input_tokens == session_input_tokens
print("Parent span:", parent_span.name)
print("Session trace ID:", format(parent_span.context.trace_id, "032x"))
print("Nested agent runs:", session_run_count)
print("Total input tokens:", session_input_tokens)
print("Total output tokens:", session_output_tokens)
print("Input tokens independently summed from agent spans:", span_input_tokens)

18:04:00.810 trivia_session
18:04:00.811   agent run
18:04:00.813     chat gpt-4o-mini


Logfire project URL: ]8;id=804997;https://logfire-us.pydantic.dev/priyagarnepudi/priya-logfire-test\https://logfire-us.pydantic.dev/priyagarnepudi/priya-logfire-test]8;;\

18:04:02.360     running 1 tool
18:04:02.360       running tool: get_categories


18:04:02.769     chat gpt-4o-mini


18:04:03.842     running 1 tool
18:04:03.843       running tool: get_questions


18:04:04.188     chat gpt-4o-mini


... (3 outputs left out)

Quizmaster (run 4):
That's not correct. The correct answer is **B) Hydrogen**.

Hydrogen is the most abundant element in the universe, making up about 74% of its elemental mass. It plays a vital role in the formation of stars and is primarily found in stars, where it undergoes nuclear fusion to form helium and release energy, powering the stars.

Now, let's proceed to the next question!

**Question 4:** Human cells typically have how many copies of each gene?  
A) 1  
B) 2  
C) 3  
D) 4  

What is your answer?

You: A
18:04:14.967   agent run
18:04:14.968     chat gpt-4o-mini


Quizmaster (run 5):
That's not correct. The correct answer is **B) 2**.

Human cells usually have two copies of each gene, because humans are diploid organisms. This means that we inherit one copy of each gene from our mother and one from our father, giving our cells a complete set of chromosomes that includes two copies of each gene.

Let's move on to the final question!

**Question 5:** What did Gregory Mendel use to test genetic crossovers?  
A) Cats  
B) Peas  
C) Flowers  
D) Parrots  

What is your answer?

You: A
18:04:17.533   agent run
18:04:17.534     chat gpt-4o-mini


Quizmaster (run 6):
That's not correct. The correct answer is **B) Peas**.

Gregor Mendel, known as the father of genetics, conducted his groundbreaking experiments on inheritance using pea plants. He studied the traits of pea plants, such as color and shape, and through these experiments, he formulated the laws of inheritance, which laid the foundation for genetics as we know it today.

### Final Score
You answered 0 out of 5 questions correctly. 

Don't be discouraged; trivia is all about learning! If you'd like, we can try again or explore another category of questions. What would you like to do next?



Parent span: trivia_session
Session trace ID: 01a1ae993daa13a5367adec49a79901f
Nested agent runs: 6
Total input tokens: 5452
Total output tokens: 684
Input tokens independently summed from agent spans: 5452


### Verified answer

**Approximately 5,500 input tokens** (exactly **5,452** for this completed session).

- Parent: `trivia_session`
- Nested agent runs: **6**
- Total input tokens: **5,452**
- Total output tokens: **684**
- Trace ID: `01a1ae993daa13a5367adec49a79901f`

The run completed all five questions and returned a final score. The sum of `result.usage().input_tokens` across all six runs matched the sum of `gen_ai.usage.input_tokens` on the captured agent spans. All six agent spans had the session span as their direct parent.

Cloud export was configured with the supplied write token. This count was independently verified from locally captured spans; cloud dashboard verification still requires a valid read token or signed-in dashboard.

To check this trace in Logfire, sum only agent-run spans to avoid double-counting the model-call child spans:

```sql
SELECT SUM((attributes->>'gen_ai.usage.input_tokens')::BIGINT) AS input_tokens
FROM records
WHERE trace_id = '01a1ae993daa13a5367adec49a79901f'
  AND span_name IN ('agent run', 'invoke_agent agent');
```

Your total may differ because the trivia questions and model-generated explanations vary. The provided parent span does not automatically acquire a token total attribute; compute the sum across its agent-run children or use the dashboard's descendant aggregation.

## Question 5: Get trace IDs with SQL

**Grouped session trace ID from Q4:**

```text
01a1ae993daa13a5367adec49a79901f
```

This ID was captured from the actual `trivia_session` parent span during Q4; it is not an invented example. A cloud SQL lookup was attempted, but the configured read token returned **Invalid token**. Therefore the programmatic cloud lookup and UI cross-check remain unverified. Replace `LOGFIRE_READ_TOKEN` in `.env` with a valid read token from the same project receiving the write-token exports. Never paste tokens into the submitted notebook.

The following cell queries grouped sessions and checks for the specific Q4 trace. It has not been executed successfully with the current read token.

In [ ]:
import os
from dotenv import load_dotenv
from logfire.query_client import LogfireQueryClient

load_dotenv(project_root / ".env", override=True)
client = LogfireQueryClient(read_token=os.environ["LOGFIRE_READ_TOKEN"])

session_rows = client.query_json_rows(
    sql="""
    SELECT trace_id, span_name, start_timestamp, duration
    FROM records
    WHERE service_name = 'trivia-agent'
      AND span_name = 'trivia_session'
    ORDER BY start_timestamp DESC
    LIMIT 100
    """
)
trace_ids = [row["trace_id"] for row in session_rows["rows"]]
print("Grouped session trace IDs:", trace_ids)

q4_trace_id = "01a1ae993daa13a5367adec49a79901f"
q4_sessions = [row for row in session_rows["rows"] if row["trace_id"] == q4_trace_id]
assert q4_sessions, "Q4 trace not found: check the project, region, token, and retention window"
print("Q4 grouped session:", q4_sessions[0])

### Cross-check in the Logfire UI

Open the same project receiving the agent's write-token exports, find the `trivia_session` span, and compare its trace ID with `01a1ae993daa13a5367adec49a79901f`. Expanding the session should show six agent-run spans nested under it. Select a time range that includes the recorded session. The trace ID is shared by the parent and all child spans, whereas each span has its own span ID.

## Question 6: Reconstruct runs and calculate costs

**Answer: approximately $0.00123 (about 0.12 US cents).**

The grouped Q4 session used **5,452 input tokens** and **684 output tokens**, measured during the completed session and independently summed from its agent-run spans.

### Replay every agent run's usage

The project's existing `trace_replay` module already matches both `agent run` and `invoke_agent <name>`. It has now been corrected to fetch every matching span and sum their token counts, instead of selecting only the latest span. It preserves the latest span's cumulative conversation history for reconstruction, avoiding duplication of previous turns. This assumes the history is passed forward across turns, as in our session. It does not merge unrelated conversations sharing a trace.

The regression tests verify aggregation for both `fetch_trace` and `fetch_traces`; the complete replay tests passed: **17 passed**.

The cloud replay cell below requires a valid read token from the same project. It remains unexecuted because the current read token was rejected in Question 5. The cost calculation afterward uses the independently captured Q4 counts; no cloud replay success is claimed.

In [ ]:
import trace_replay

# Requires the valid query client from Question 5.
grouped_trace = trace_replay.fetch_trace(
    "01a1ae993daa13a5367adec49a79901f", client
)
grouped_run = trace_replay.trace_to_run_result(grouped_trace)
print("Reconstructed final answer:", grouped_run.output)
print("Summed input tokens:", grouped_run.usage().input_tokens)
print("Summed output tokens:", grouped_run.usage().output_tokens)

### Calculate the cost from verified Q4 usage

Use the pricing supplied in the homework. This treats all input tokens at the standard input rate, without cached-input discounts. The estimate covers the complete grouped session, not only its last turn.

In [ ]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}


def calculate_cost(model_name, input_tokens, output_tokens):
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost


# Actual Q4 counts; replace with grouped_run.usage() values after cloud replay.
input_tokens = 5452
output_tokens = 684
session_cost = calculate_cost("gpt-4o-mini", input_tokens, output_tokens)
print(f"Input cost: ${input_tokens / 1_000_000 * 0.15:.7f}")
print(f"Output cost: ${output_tokens / 1_000_000 * 0.60:.7f}")
print(f"Total cost: ${session_cost:.7f}")
print(f"Approximate cost: ${session_cost:.5f} ({session_cost * 100:.3f} cents)")

Input cost: $0.0008178
Output cost: $0.0004104
Total cost: $0.0012282
Approximate cost: $0.00123 (0.123 cents)


Calculation: `(5452 × 0.15 + 684 × 0.60) / 1,000,000 = $0.0012282`. Round to **$0.0012** for an approximate homework answer. Different randomly generated questions and explanations can produce different token counts and costs.

## Question 7 (Bonus): Feedback tracking

**Correct choice: By using `logfire.attach_context()` with the session context.**

The context carries the trace and parent span identifiers. Attaching it when recording the feedback event associates that event with the same session trace. Merely storing feedback in the agent output, changing the Logfire project, or switching the log level does not establish that relationship.

When feedback is logged synchronously inside the still-active `logfire.span("trivia_session")`, it already inherits the session context. Explicit `attach_context` is especially useful if feedback is submitted later from a different callback or execution context.

Installed `questionary` with `uv add questionary`. The terminal application now wraps each session in `trivia_session`, preserves the session context, asks for feedback after the conversation loop, records `1` or `-1`, and skips cancelled or skipped feedback.

```python
import questionary


def ask_feedback():
    result = questionary.select(
        "How was the trivia session?",
        choices=["👍 Good", "👎 Bad", "Skip"],
    ).ask()
    if result is None or result == "Skip":
        return None
    return 1 if "Good" in result else -1


with logfire.span("trivia_session"):
    session_context = logfire.get_context()
    run("Let's play 5 easy questions from Science & Nature")
    feedback = ask_feedback()
    if feedback is not None:
        with logfire.attach_context(session_context):
            logfire.info("user_feedback", feedback=feedback)
logfire.force_flush()
```

### Record real feedback

Run this command in a terminal, answer the quiz questions, type `stop`, and select your rating. Repeat for additional sessions:

```bash
uv run python trivia_agent.py "Let's play 5 easy questions from Science & Nature"
```

The interactive `questionary` prompt is intended for the terminal. No extra sessions or user ratings were fabricated here. Feedback counts remain pending actual user ratings and a valid read token.

In [ ]:
# Requires the valid Logfire query client from Question 5.
feedback_counts = client.query_json_rows(
    sql="""
    SELECT
        CASE (attributes->>'feedback')::INTEGER
            WHEN 1 THEN 'positive'
            WHEN -1 THEN 'negative'
        END AS rating,
        COUNT(*) AS feedback_count
    FROM records
    WHERE service_name = 'trivia-agent'
      AND span_name = 'user_feedback'
      AND (attributes->>'feedback')::INTEGER IN (1, -1)
    GROUP BY rating
    ORDER BY rating
    """
)
print(feedback_counts["rows"])

To verify the association in Logfire, open the `user_feedback` event and compare its trace ID with its session's `trivia_session` span. They should match. The event has its own span ID, but shares the trace ID with the quiz's agent runs.